# 12 · Final project: your own neural network, end to end

## What you will build

A complete model-training pipeline around your own network: a train/validation/test split, feature standardisation, training with the epoch chosen on validation data, a single test evaluation, saving and reloading the trained parameters, and a written account of what the model can and cannot do.

## Requirements checklist

- [ ] Load a dataset (spirals by default; the breast-cancer data is optional).
- [ ] Split it into training, validation and test sets **before** fitting anything.
- [ ] Standardise features with statistics from the training set only.
- [ ] Train your network; plot training and validation loss.
- [ ] Keep the epoch with the best validation loss.
- [ ] Evaluate the kept model on the test set **once**.
- [ ] Save the parameters, reload them, and confirm identical predictions.
- [ ] Explain the model's limitations in your own words.

## Dependencies

Tier 1: NumPy and Matplotlib. The optional breast-cancer data is loaded with scikit-learn in a cell tagged `data`; nothing else uses it.

## Setup

Run this cell first. It loads NumPy and Matplotlib and defines `practice`, the checker used after every ✍️ cell.

In [ ]:
# ── Setup ─ run this cell first ────────────────────────────────────────────
import numpy as np

rng = np.random.default_rng(0)


def _close(got, want, atol):
    if isinstance(want, dict):
        return isinstance(got, dict) and got.keys() == want.keys() and all(
            _close(got[key], want[key], atol) for key in want
        )
    if isinstance(want, (list, tuple)):
        return isinstance(got, (list, tuple)) and len(got) == len(want) and all(
            _close(g, w, atol) for g, w in zip(got, want)
        )
    try:
        got_array, want_array = np.asarray(got, dtype=float), np.asarray(want, dtype=float)
    except (TypeError, ValueError):
        return got == want
    return got_array.shape == want_array.shape and np.allclose(got_array, want_array, atol=atol)


def _short(value):
    if isinstance(value, tuple):
        return "(" + ", ".join(_short(item) for item in value) + ")"
    if isinstance(value, np.ndarray):
        text = np.array2string(value, precision=4, threshold=8, separator=", ").replace("\n", "")
    else:
        text = repr(value)
    return text if len(text) < 120 else text[:117] + "..."


def practice(name, yours, reference, cases, atol=1e-6):
    """Check your function against the reference on a few inputs.

    Returns the function the rest of the notebook should use: yours once it
    matches, otherwise the reference -- so you can always keep going.
    """
    try:
        for args in cases:
            got, want = yours(*args), reference(*args)
            if not _close(got, want, atol):
                print(f"❌ {name}: for input {_short(args)}")
                print(f"   you returned {_short(got)}")
                print(f"   expected     {_short(want)}")
                print(f"   The rest of the notebook uses the reference {name} until yours matches.")
                return reference
    except NotImplementedError:
        print(f"✍️  {name}: not written yet. Type it in the cell above and run this cell again.")
        print("   Until then the rest of the notebook uses the reference version.")
        return reference
    except Exception as error:  # a typo in the learner's code must not stop the lesson
        print(f"❌ {name} raised {type(error).__name__}: {error}")
        print(f"   The rest of the notebook uses the reference {name} until yours runs.")
        return reference
    print(f"✅ {name}: matches the reference on {len(cases)} test case{'s' if len(cases) != 1 else ''}. From here on the notebook runs YOUR {name}.")
    return yours

import matplotlib.pyplot as plt

## 📦 Recap: what you built earlier

The hidden cell below defines the pieces from earlier notebooks under their usual names, so this notebook runs on its own. Open it if you want to see them again.

In [ ]:
# 📦 Recap: the pieces from notebooks 01–04, under their usual names.

def relu(Z):
    """ReLU, element by element: max(0, Z)."""
    return np.maximum(0.0, Z)


def relu_grad(Z):
    """The slope of ReLU: 1.0 where Z > 0, else 0.0 (we take 0 at Z = 0)."""
    return (Z > 0).astype(float)


def sigmoid(Z):
    """Numerically stable logistic function 1 / (1 + exp(-Z)), element by element."""
    return np.exp(-np.logaddexp(0.0, -Z))


def softmax(Z):
    """Row-wise softmax of logits Z (batch, classes). Subtracting each row's maximum avoids overflow."""
    exps = np.exp(Z - Z.max(axis=1, keepdims=True))
    return exps / exps.sum(axis=1, keepdims=True)


def dense_forward(X, W, b):
    """Z = X @ W + b. X: (batch, n_in), W: (n_in, n_out), b: (n_out,) -> Z: (batch, n_out)."""
    return X @ W + b


def init_network(sizes, rng):
    """He-initialised layers for sizes such as [2, 64, 3]: a list of {"W": (n_in, n_out), "b": (n_out,)}."""
    return [
        {"W": rng.normal(0.0, np.sqrt(2.0 / n_in), (n_in, n_out)), "b": np.zeros(n_out)}
        for n_in, n_out in zip(sizes[:-1], sizes[1:])
    ]


def forward(network, X):
    """ReLU hidden layers and a softmax output.

    Returns (probs, cache): probs has shape (batch, classes), and cache[i] = (A_prev, Z) for layer i.
    """
    cache = []
    A = X
    for index, layer in enumerate(network):
        Z = dense_forward(A, layer["W"], layer["b"])
        cache.append((A, Z))
        A = softmax(Z) if index == len(network) - 1 else relu(Z)
    return A, cache


def predict(network, X):
    """The predicted class index for every row of X: shape (batch,)."""
    probs, _ = forward(network, X)
    return probs.argmax(axis=1)


def accuracy(network, X, y):
    """The fraction of rows of X whose predicted class equals the label in y."""
    return float(np.mean(predict(network, X) == y))


def make_spirals(n_per_class, n_classes, noise, rng):
    """n_classes interleaved spiral arms in 2-D, sorted by class.

    Returns X with shape (n_per_class * n_classes, 2) and integer labels y with
    shape (n_per_class * n_classes,). noise is the spread of the random angle jitter.
    """
    radius = np.linspace(0.05, 1.0, n_per_class)
    X_parts, y_parts = [], []
    for k in range(n_classes):
        angle = 2 * np.pi * k / n_classes + 6.0 * radius + rng.normal(0.0, noise, n_per_class)
        X_parts.append(np.column_stack([radius * np.cos(angle), radius * np.sin(angle)]))
        y_parts.append(np.full(n_per_class, k))
    return np.concatenate(X_parts), np.concatenate(y_parts)


# From notebook 05: loss functions.

def mse(pred, target):
    """Mean squared error over every entry of two arrays with the same shape."""
    return float(np.mean((pred - target) ** 2))


def binary_cross_entropy(p, y):
    """Mean binary cross-entropy of probabilities p (batch,) against 0/1 labels y (batch,), clipped."""
    p = np.clip(p, 1e-12, 1 - 1e-12)
    return float(-np.mean(y * np.log(p) + (1 - y) * np.log(1 - p)))


def cross_entropy(probs, y):
    """Mean negative log-probability of the true class. probs: (batch, classes), y: (batch,) ints."""
    picked = probs[np.arange(len(y)), y]
    return float(-np.mean(np.log(np.clip(picked, 1e-12, None))))


def cross_entropy_from_logits(Z, y):
    """The same loss computed from logits Z (batch, classes) with the log-sum-exp trick."""
    row_max = Z.max(axis=1, keepdims=True)
    log_sum_exp = row_max[:, 0] + np.log(np.exp(Z - row_max).sum(axis=1))
    return float(np.mean(log_sum_exp - Z[np.arange(len(y)), y]))


# From notebook 06: numerical derivatives and gradient checking.

def numerical_derivative(f, x, h=1e-5):
    """Central-difference estimate of f'(x) for a function of one number."""
    return (f(x + h) - f(x - h)) / (2 * h)


def numerical_gradient(f, array, eps=1e-5):
    """Central-difference gradient of the scalar f(array) with respect to every entry of array.

    Each entry is nudged in place and then restored, so f may also read the array
    through a closure (for example a network's weights). Returns an array shaped like array.
    """
    grad = np.zeros(array.shape)
    for index in np.ndindex(array.shape):
        original = array[index]
        array[index] = original + eps
        loss_plus = f(array)
        array[index] = original - eps
        loss_minus = f(array)
        array[index] = original
        grad[index] = (loss_plus - loss_minus) / (2 * eps)
    return grad


def relative_error(a, b):
    """||a - b|| / (||a|| + ||b||), with a tiny floor on the denominator: 0 when equal, at most 1."""
    a, b = np.asarray(a, dtype=float), np.asarray(b, dtype=float)
    return float(np.linalg.norm(a - b) / max(np.linalg.norm(a) + np.linalg.norm(b), 1e-12))

# From notebook 07: backpropagation.

def softmax_cross_entropy_grad(probs, y):
    """dL/dZ for softmax + mean cross-entropy: (P - Y) / n."""
    return (probs - np.eye(probs.shape[1])[y]) / len(y)


def dense_backward(dZ, A_prev, W):
    """Gradients of a dense layer: (dW, db, dA_prev)."""
    return A_prev.T @ dZ, dZ.sum(axis=0), dZ @ W.T


def backward(network, cache, probs, y):
    """Every layer's gradients, last layer first, returned in layer order."""
    grads = [None] * len(network)
    dZ = softmax_cross_entropy_grad(probs, y)
    for i in reversed(range(len(network))):
        A_prev, Z = cache[i]
        if i < len(network) - 1:
            dZ = dA * relu_grad(Z)
        dW, db, dA = dense_backward(dZ, A_prev, network[i]["W"])
        grads[i] = {"W": dW, "b": db}
    return grads


# From notebook 08: training with gradient descent.

def sgd_step(network, grads, lr):
    """A new network one gradient-descent step downhill: W - lr * dW, b - lr * db."""
    return [{"W": layer["W"] - lr * g["W"], "b": layer["b"] - lr * g["b"]} for layer, g in zip(network, grads)]


def minibatch_indices(n, batch_size, seed):
    """A shuffled split of range(n) into batches of batch_size (the last may be smaller)."""
    order = np.random.default_rng(seed).permutation(n)
    return [order[start:start + batch_size] for start in range(0, n, batch_size)]


def train_epoch(network, X, y, lr, batch_size, seed):
    """One pass over the data in shuffled mini-batches. Returns (network, mean batch loss)."""
    losses = []
    for batch in minibatch_indices(len(X), batch_size, seed):
        probs, cache = forward(network, X[batch])
        losses.append(cross_entropy(probs, y[batch]))
        network = sgd_step(network, backward(network, cache, probs, y[batch]), lr)
    return network, float(np.mean(losses))


def train(network, X, y, X_val, y_val, lr, epochs, batch_size, seed=0):
    """Train for some epochs; returns (network, history of losses and accuracies per epoch)."""
    history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
    for epoch in range(epochs):
        network, _ = train_epoch(network, X, y, lr, batch_size, seed + epoch)
        history["train_loss"].append(cross_entropy(forward(network, X)[0], y))
        history["val_loss"].append(cross_entropy(forward(network, X_val)[0], y_val))
        history["train_acc"].append(accuracy(network, X, y))
        history["val_acc"].append(accuracy(network, X_val, y_val))
    return network, history

## 1 · Data

The default task is the three-class spirals. To use real measurements instead, run the optional cell after it: scikit-learn's bundled breast-cancer data (569 tumours, 30 measurements, benign or malignant), loaded without a download.

In [ ]:
X_data, y_data = make_spirals(250, 3, 0.25, np.random.default_rng(20))
dataset_name = "spirals"
print(dataset_name, X_data.shape, np.bincount(y_data))

In [ ]:
# Optional: use real data instead. Uncomment the last line to switch.
from sklearn.datasets import load_breast_cancer

cancer = load_breast_cancer()
# X_data, y_data, dataset_name = cancer.data, cancer.target, "breast cancer"

## 2 · Split before you look

Decide the three roles first. The **training** set fits the weights, the **validation** set makes choices (here: which epoch to keep), and the **test** set is used once at the end to estimate how the chosen model does on new data. Splitting per class (stratifying) keeps the class proportions the same in every part.

### ✍️ Your turn: `train_val_test_split`

Shuffle the indices of each class with `rng = np.random.default_rng(seed)` (classes in `np.unique(y)` order, one `rng.permutation` per class), cut each class into `fractions` (e.g. `(0.6, 0.2, 0.2)`) with `int(round(...))` for the first two parts, and return `X_train, y_train, X_val, y_val, X_test, y_test`. Concatenate the parts class by class.

Type it rather than pasting it: typing is how the shapes and indices sink in. The reference is in the collapsed cell below if you get stuck.

In [ ]:
def train_val_test_split_reference(X, y, fractions, seed):
    rng = np.random.default_rng(seed)
    parts = ([], [], [])
    for label in np.unique(y):
        rows = rng.permutation(np.flatnonzero(y == label))
        n_train = int(round(fractions[0] * len(rows)))
        n_val = int(round(fractions[1] * len(rows)))
        for part, chunk in zip(parts, (rows[:n_train], rows[n_train:n_train + n_val], rows[n_train + n_val:])):
            part.append(chunk)
    train, val, test = (np.concatenate(part) for part in parts)
    return X[train], y[train], X[val], y[val], X[test], y[test]

In [ ]:
def train_val_test_split(X, y, fractions, seed):
    """Stratified split into training, validation and test sets."""
    # ✍️ Your code here
    raise NotImplementedError

In [ ]:
toy_X = np.arange(40, dtype=float).reshape(20, 2)
toy_y = np.array([0] * 10 + [1] * 10)
train_val_test_split = practice("train_val_test_split", train_val_test_split, train_val_test_split_reference, cases=[
    (toy_X, toy_y, (0.6, 0.2, 0.2), 0),
    (toy_X, toy_y, (0.5, 0.3, 0.2), 3),
])

In [ ]:
X_train, y_train, X_val, y_val, X_test, y_test = train_val_test_split(X_data, y_data, (0.6, 0.2, 0.2), seed=1)
print("train", X_train.shape, "validation", X_val.shape, "test", X_test.shape)

### ✍️ Your turn: `fit_standardizer`

Return `(mean, std)` of each column of `X_train` (`axis=0`), with any zero standard deviation replaced by 1.0 so constant columns do not divide by zero.

Type it rather than pasting it: typing is how the shapes and indices sink in. The reference is in the collapsed cell below if you get stuck.

In [ ]:
def fit_standardizer_reference(X_train):
    mean = X_train.mean(axis=0)
    std = X_train.std(axis=0)
    return mean, np.where(std == 0, 1.0, std)

In [ ]:
def fit_standardizer(X_train):
    """Column means and standard deviations from the training set only."""
    # ✍️ Your code here
    raise NotImplementedError

In [ ]:
fit_standardizer = practice("fit_standardizer", fit_standardizer, fit_standardizer_reference, cases=[
    (np.array([[1.0, 5.0], [3.0, 5.0], [5.0, 5.0]]),),
    (np.random.default_rng(0).normal(3, 2, size=(50, 4)),),
])
mean, std = fit_standardizer(X_train)
X_train_s, X_val_s, X_test_s = ((part - mean) / std for part in (X_train, X_val, X_test))

## 3 · Your `NeuralNetwork` class

The class below packages the functions you wrote into one object with `fit`, `predict_proba`, `predict`, `evaluate`, `save` and `load`. Read it: every method is a few lines that call your own code. `fit` keeps the parameters from the epoch with the lowest validation loss (early stopping by checkpoint).

In [ ]:
class NeuralNetwork:
    def __init__(self, sizes, seed=0):
        self.sizes = list(sizes)
        self.layers = init_network(self.sizes, np.random.default_rng(seed))

    def predict_proba(self, X):
        return forward(self.layers, X)[0]

    def predict(self, X):
        return self.predict_proba(X).argmax(axis=1)

    def evaluate(self, X, y):
        probs = self.predict_proba(X)
        return {"loss": cross_entropy(probs, y), "accuracy": float(np.mean(probs.argmax(axis=1) == y))}

    def fit(self, X, y, X_val, y_val, lr=0.5, epochs=100, batch_size=32, seed=0):
        history = {"train_loss": [], "val_loss": []}
        best_loss, best_layers, self.best_epoch = np.inf, self.layers, 0
        for epoch in range(epochs):
            self.layers, _ = train_epoch(self.layers, X, y, lr, batch_size, seed + epoch)
            history["train_loss"].append(self.evaluate(X, y)["loss"])
            history["val_loss"].append(self.evaluate(X_val, y_val)["loss"])
            if history["val_loss"][-1] < best_loss:
                best_loss, best_layers, self.best_epoch = history["val_loss"][-1], self.layers, epoch
        self.layers = best_layers
        return history

    def save(self, path):
        arrays = {f"{name}{i}": layer[name] for i, layer in enumerate(self.layers) for name in ("W", "b")}
        np.savez(path, sizes=np.array(self.sizes), **arrays)

    @classmethod
    def load(cls, path):
        data = np.load(path)
        model = cls([int(size) for size in data["sizes"]])
        model.layers = [{"W": data[f"W{i}"], "b": data[f"b{i}"]} for i in range(len(model.sizes) - 1)]
        return model

## 4 · Train, choose, test once

🧪 Change the hidden sizes, learning rate or epochs and re-run. Keep notes of what you tried: every decision must be made on the validation set.

In [ ]:
n_classes = int(y_data.max()) + 1
model = NeuralNetwork([X_train_s.shape[1], 64, 64, n_classes], seed=7)
history = model.fit(X_train_s, y_train, X_val_s, y_val, lr=0.5, epochs=100, batch_size=32)
print(f"best validation loss at epoch {model.best_epoch}")
plt.figure(figsize=(5, 3))
plt.plot(history["train_loss"], label="training"); plt.plot(history["val_loss"], label="validation")
plt.axvline(model.best_epoch, ls=":", c="grey", label="kept epoch")
plt.xlabel("epoch"); plt.ylabel("cross-entropy"); plt.legend(); plt.show()

Only now, with every choice made, look at the test set, once. Compare it with the simplest baseline: always predicting the most common training class.

In [ ]:
test = model.evaluate(X_test_s, y_test)
baseline = float(np.mean(y_test == np.bincount(y_train).argmax()))
print(f"{dataset_name}: test accuracy {test['accuracy']:.3f} (baseline {baseline:.3f}), test loss {test['loss']:.3f}")

## 5 · Save, reload, compare

In [ ]:
model.save("my_network.npz")
reloaded = NeuralNetwork.load("my_network.npz")
same = np.array_equal(model.predict(X_test_s), reloaded.predict(X_test_s))
print("reloaded model gives identical predictions:", same)

## 6 · Put it together

In [ ]:
assert len(X_train) + len(X_val) + len(X_test) == len(X_data)
assert np.allclose(X_train_s.mean(axis=0), 0, atol=1e-9), "training features should be centred"
assert same, "the saved and reloaded model must predict identically"
assert test["accuracy"] > baseline + 0.2, "the network should clearly beat the majority-class baseline"
print("✅ Pipeline complete: split, scaled, trained with early stopping, tested once, saved and reloaded.")

## 7 · Your write-up

Answer in this cell (double-click to edit):

1. **Data and protocol.** Which dataset, how was it split, and which choices did the validation set make?
2. **Result.** Test accuracy against the baseline, with the number of test examples. How much would one more wrong prediction change it?
3. **Three mistakes.** Look at three misclassified test examples. What do they have in common?
4. **Limitations.** What does this model *not* show? (For example: a small generated or single-source dataset, one random seed, no claim about other populations, and for the cancer data: not medical advice.)

## Self-review rubric

- No statistic used for scaling or model choice came from the test set.
- Validation chose the epoch; the test set was evaluated once.
- The reloaded model reproduces the predictions exactly.
- The write-up states the baseline, the sample size and at least one failure case.

## Common errors and debugging

- **Test accuracy far below validation**: a decision leaked from the test set, or the split is not stratified.
- **Scaling with all the data**: fit the standardiser on `X_train` only, then apply it to every part.
- **`load` fails**: `np.savez` adds `.npz` to the file name if it is missing; load the same name you saved.

## What to learn next

**Base Model Training** moves to the engineering of training at scale: data pipelines, optimisers, checkpoints and evaluation, with a framework that computes the gradients you derived by hand automatically. The transformer, language, vision and speech courses build on it.